In [ ]:
%pip install transformers datasets torch scikit-learn evaluate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# pip install --upgrade torch

In [2]:
# %pip install 

In [3]:
# pip show torch

In [4]:
# %pip install --upgrade transformers

In [ ]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from model import *
from transformers import TrainingArguments
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding

In [9]:
sys.path.append('.')

In [11]:

def main_function(mode="multiclass"):
    if mode == "binary":
        num_labels = 2
        model_class = BertweetModelBinary
        train_path = "\data\processed_data\test_binary_preprocessed.csv"
        val_path = "\data\processed_data\validation_binary_preprocessed.csv"
        test_path = "\data\processed_data\test_binary_preprocessed.csv"
        output_dir = "./results_binary"
        logging_dir = "./logs_binary"
    else:
        num_labels = 5
        model_class = BertweetModelMulticlass
        train_path = "\data\processed_data\train_multiclass_balanced.csv"
        val_path = "\data\processed_data\validation_multiclass_preprocessed.csv"
        test_path = "\data\processed_data\test_multiclass_preprocessed.csv"
        output_dir = "./results_multiclass"
        logging_dir = "./logs_multiclass"

    
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)


    train_df = train_df.dropna(subset=["tweet_soft"])
    val_df = val_df.dropna(subset=["tweet_soft"])
    test_df = test_df.dropna(subset=["tweet_soft"])

    ## Convert to huggingface dataset
    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)
    test_dataset = Dataset.from_pandas(test_df)


    ## Model initialization
    classifier = model_class(num_labels=num_labels, model_name="vinai/bertweet-base")

    print("Device being used:", classifier.device)
    print(f"model {classifier}")


    # Preprocess data
    tokenized_train_dataset = classifier.preprocess_data(train_dataset)
    tokenized_val_dataset = classifier.preprocess_data(val_dataset)
    tokenized_test_dataset = classifier.preprocess_data(test_dataset)

    # print("Tokenizers: ",tokenized_train_dataset.unique("label"))

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,  
        per_device_train_batch_size=16, 
        per_device_eval_batch_size=16,
        num_train_epochs=5,  
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_dir=logging_dir,
        logging_strategy="epoch",   
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_weighted",
        fp16=True,
        warmup_ratio=0.1,
        logging_first_step=True,
        greater_is_better=True,
        report_to="none",
        gradient_accumulation_steps=2,  
        max_grad_norm=1.0,  
        dataloader_drop_last=False,
        lr_scheduler_type="cosine_with_restarts",
    )

     # early stopping callback
    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.001
    )

    data_collator = DataCollatorWithPadding(tokenizer=classifier.tokenizer)

    # Train with callback
    classifier.train(
        tokenized_train_dataset,
        tokenized_val_dataset,
        training_args,
        callbacks=[early_stopping],
        data_collator=data_collator
    )

    # Evaluation
    classifier.plot_confusion_matrix(tokenized_val_dataset)
    eval_results = classifier.evaluate(tokenized_val_dataset)
    classifier.print_metrics_summary(eval_results)

    classifier.save_model(output_dir)

    return classifier, eval_results

In [12]:
# main_function("binary")

In [3]:
import torch

### Loading the tokenizer, Model, and doing the predictions

In [4]:
# # Load the merged model - only for multiclass classification
# model_path = "./results/merged_results_multiclass"

# # Load the base model using AutoModelForSequenceClassification
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# # Determine the number of labels (hardcoded for multiclass in this block)
# num_labels = 5

# # Load the tokenizer separately as it was saved
# loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

# # Load the model using the appropriate class from transformers
# loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)

# # Initialize the classifier with the loaded model and tokenizer
# classifier = BertweetModelMulticlass(num_labels=num_labels, model_name="vinai/bertweet-base")

# # Replace the internal model and tokenizer with the loaded ones
# classifier.model = loaded_model
# classifier.tokenizer = loaded_tokenizer

# # Ensure the model is on the correct device
# classifier.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# classifier.model.to(classifier.device)

# text = "This is black ass nigga"
# predicted, probs = classifier.predict_single(text)
# print("Predicted Class:", predicted.item())
# print("Probabilities:", probs.cpu().numpy())

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Predicted Class: 1
Probabilities: [[4.2159582e-04 9.7352231e-01 9.8710097e-03 1.4819596e-02 1.3654925e-03]]


In [8]:
# # Load the merged model - only for binary classification
# model_path = "./results/merged_results_binary"

# # Load the base model using AutoModelForSequenceClassification
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# # Determine the number of labels (hardcoded for binary in this block)
# num_labels = 2

# # Load the tokenizer separately as it was saved
# loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

# # Load the model using the appropriate class from transformers
# loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)

# # Initialize the classifier with the loaded model and tokenizer
# classifier = BertweetModelBinary(num_labels=num_labels, model_name="vinai/bertweet-base")

# # Replace the internal model and tokenizer with the loaded ones
# classifier.model = loaded_model
# classifier.tokenizer = loaded_tokenizer

# # Ensure the model is on the correct device
# classifier.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# classifier.model.to(classifier.device)

# text = "your religion is a lie, you are a slave to the system"
# predicted, probs = classifier.predict_single(text)
# print("Predicted Class:", predicted.item())
# print("Probabilities:", probs.cpu().numpy())

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Predicted Class: 1
Probabilities: [[0.1835677  0.81643224]]
